## v10 — Base-only 최적화 (OOF 7점대 목표)

### 반영한 방향
- (1) `USE_LOG_TARGET=False` + MAE 목적함수 통일
  - LightGBM: `objective='regression_l1'`
  - CatBoost: `loss_function='MAE'`
  - XGBoost: 가능하면 `objective='reg:absoluteerror'` 사용(미지원이면 안전 폴백)
- (2) 타겟 lag 없이 시계열 피처 확장
  - 주요 운영 피처에 대해 scenario별 lag/rolling/차분 추가
- (3) 약한 모델 제거/가중치 강화
  - 모델별 OOF MAE 출력
  - `WEIGHT_POWER`로 가중치 차등 (w=1/MAE^p)
  - `DROP_WORST_IF_GAP_GT` 조건으로 최약 모델 자동 제외 옵션

### 출력
- 모델별 OOF MAE
- 앙상블 OOF MAE
- 제출 파일: `submission_v10_base_only.csv` (프로젝트 루트 저장)


In [ ]:
import os
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import catboost as cb

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error

warnings.filterwarnings('ignore')

# ============================================================
# 실행/튜닝 파라미터
# ============================================================
N_FOLDS = 5
SEED = 42

# (1) 원스케일 MAE
USE_LOG_TARGET = False

# (3) 앙상블 가중치 강화: w = 1/mae^p
WEIGHT_POWER = 2.0

# (3) 약한 모델 자동 제외(옵션)
AUTO_DROP_WORST = True
DROP_WORST_IF_GAP_GT = 0.08

# 예측값 후처리
CLIP_PRED_MIN = 0.0
CLIP_PRED_MAX_Q = 0.995


def elapsed(start):
    s = int(time.time() - start)
    return f"{s//60}m {s%60:02d}s"


def section(title):
    print(f"\n{'='*60}\n  {title}\n{'='*60}")


def _resolve_data_dir() -> str:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        d = p / 'data'
        if d.is_dir() and (d / 'train.csv').is_file():
            return str(d)
    raise FileNotFoundError('data/train.csv 를 찾을 수 없습니다. 프로젝트 루트에서 실행하세요.')


def to_train_target(y):
    return np.log1p(y) if USE_LOG_TARGET else y


def from_train_pred(p):
    return np.expm1(p) if USE_LOG_TARGET else p


def mae(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)


# ============================================================
# 1) 데이터 로드
# ============================================================
path = _resolve_data_dir()
project_root = str(Path(path).resolve().parent)
print(f"▶ data dir: {path}")
print(f"▶ project root: {project_root}")

TARGET = 'avg_delay_minutes_next_30m'
ID_COLS = ['ID', 'layout_id', 'scenario_id']

t0 = time.time()
train = pd.read_csv(os.path.join(path, 'train.csv'))
test  = pd.read_csv(os.path.join(path, 'test.csv'))
layout = pd.read_csv(os.path.join(path, 'layout_info.csv'))
print(f"▶ load done ({elapsed(t0)})  train {len(train):,} / test {len(test):,}")


# ============================================================
# 2) 전처리
# ============================================================
def handle_missing_values(df):
    df = df.sort_values(['scenario_id', 'ID']).reset_index(drop=True)
    cols_to_fix = [c for c in df.columns if df[c].isnull().any() and c not in (ID_COLS + [TARGET])]
    if cols_to_fix:
        df[cols_to_fix] = df.groupby('scenario_id')[cols_to_fix].ffill()
        df[cols_to_fix] = df.groupby('scenario_id')[cols_to_fix].bfill()
        df[cols_to_fix] = df[cols_to_fix].fillna(df[cols_to_fix].median())
    return df


def add_basic_features(df):
    df = df.sort_values(['scenario_id', 'ID']).reset_index(drop=True)
    df['timeslot'] = df.groupby('scenario_id').cumcount()
    df['robot_efficiency'] = df['robot_active'] / (df['robot_total'] + 1e-6)
    df['robot_density'] = df['robot_total'] / (df['floor_area_sqm'] + 1e-6)
    df['order_per_station'] = df['order_inflow_15m'] / (df['pack_station_count'] + 1e-6)
    df['robot_per_station'] = df['robot_active'] / (df['pack_station_count'] + 1e-6)
    df['cumulative_orders'] = df.groupby('scenario_id')['order_inflow_15m'].cumsum()
    df['order_pressure'] = df['cumulative_orders'] / (df['pack_station_count'] + 1e-6)
    if 'congestion_score' in df.columns:
        df['risk_index'] = df['congestion_score'] * (1 - df['robot_efficiency'])
        df['bottle_neck'] = df['order_per_station'] * df['congestion_score']
    if 'low_battery_ratio' in df.columns:
        df['battery_risk'] = df['low_battery_ratio'] * df['robot_total']
    if 'battery_mean' in df.columns and 'battery_std' in df.columns:
        df['battery_cv'] = df['battery_std'] / (df['battery_mean'] + 1e-6)
    return df


TS_COLS = [
    'order_inflow_15m', 'robot_active', 'robot_idle', 'robot_total',
    'pack_utilization', 'congestion_score', 'avg_trip_distance',
    'low_battery_ratio', 'outbound_truck_wait_min',
    'order_per_station', 'robot_efficiency', 'order_pressure',
]


def add_timeseries_features(df):
    df = df.sort_values(['scenario_id', 'ID']).reset_index(drop=True)
    for col in TS_COLS:
        if col not in df.columns:
            continue
        g = df.groupby('scenario_id')[col]
        for lag in (1, 2, 3):
            df[f'{col}_lag{lag}'] = g.shift(lag)
        df[f'{col}_diff1'] = g.shift(1) - g.shift(2)
        df[f'{col}_roll3_mean'] = g.transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
        df[f'{col}_roll5_mean'] = g.transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
        df[f'{col}_roll3_std']  = g.transform(lambda x: x.shift(1).rolling(3, min_periods=1).std().fillna(0))

    lag_cols = [c for c in df.columns if ('_lag' in c or '_diff' in c) and c not in ID_COLS]
    if lag_cols:
        df[lag_cols] = df.groupby('scenario_id')[lag_cols].ffill()
        df[lag_cols] = df.groupby('scenario_id')[lag_cols].bfill()
    return df


def preprocess_all(df, layout_df):
    df = df.merge(layout_df, on='layout_id', how='left')
    df = handle_missing_values(df)
    df = add_basic_features(df)
    df = add_timeseries_features(df)
    if 'layout_type' in df.columns:
        df['layout_type'] = pd.factorize(df['layout_type'])[0]
    return df


section('Preprocess')
t0 = time.time()
train = preprocess_all(train, layout)
test  = preprocess_all(test, layout)
print(f"▶ preprocess done ({elapsed(t0)})")


# ============================================================
# 3) 타겟 인코딩 (OOF)
# ============================================================
section('Target Encoding')
t0 = time.time()
TE_COLS = [c for c in ['layout_id', 'timeslot', 'layout_type', 'shift_hour', 'day_of_week'] if c in train.columns]
SMOOTHING = 20
kf_te = GroupKFold(n_splits=N_FOLDS)
groups_te = train['scenario_id']
for col in TE_COLS:
    te_col = f'{col}_te'
    train[te_col] = np.nan
    global_mean = train[TARGET].mean()
    for tr_idx, val_idx in kf_te.split(train, train[TARGET], groups=groups_te):
        tr_df = train.iloc[tr_idx]
        stats = tr_df.groupby(col)[TARGET].agg(['mean', 'count'])
        smooth = (stats['count'] * stats['mean'] + SMOOTHING * global_mean) / (stats['count'] + SMOOTHING)
        train.loc[val_idx, te_col] = train.iloc[val_idx][col].map(smooth).fillna(global_mean)
    stats_full = train.groupby(col)[TARGET].agg(['mean', 'count'])
    smooth_full = (stats_full['count'] * stats_full['mean'] + SMOOTHING * global_mean) / (stats_full['count'] + SMOOTHING)
    test[te_col] = test[col].map(smooth_full).fillna(global_mean)
print(f"▶ target encoding done ({elapsed(t0)})")


# ============================================================
# 4) 학습/OOF
# ============================================================
feature_cols = [c for c in train.columns if c not in ID_COLS + [TARGET]]
print(f"▶ features: {len(feature_cols)}")

y_all = to_train_target(train[TARGET].values)
groups = train['scenario_id'].values
kf = GroupKFold(n_splits=N_FOLDS)

lgb_params = dict(
    objective='regression_l1',
    n_estimators=12000,
    learning_rate=0.01,
    max_depth=10,
    num_leaves=255,
    min_child_samples=20,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.7,
    reg_alpha=0.2,
    reg_lambda=2.0,
    random_state=SEED,
    verbose=-1,
)

xgb_params = dict(
    objective='reg:absoluteerror',
    n_estimators=12000,
    learning_rate=0.01,
    max_depth=9,
    subsample=0.8,
    colsample_bytree=0.7,
    reg_alpha=0.2,
    reg_lambda=2.0,
    random_state=SEED,
    tree_method='hist',
    eval_metric='mae',
    early_stopping_rounds=300,
    verbosity=0,
)

cat_params = dict(
    iterations=12000,
    learning_rate=0.01,
    depth=9,
    l2_leaf_reg=6.0,
    bootstrap_type='MVS',
    subsample=0.8,
    colsample_bylevel=0.7,
    loss_function='MAE',
    eval_metric='MAE',
    random_seed=SEED,
    task_type='CPU',
    early_stopping_rounds=300,
)

section('Train OOF')
oof_lgb = np.zeros(len(train))
oof_xgb = np.zeros(len(train))
oof_cat = np.zeros(len(train))
models_lgb, models_xgb, models_cat = [], [], []

for fold, (tr_idx, va_idx) in enumerate(kf.split(train, y_all, groups=groups), 1):
    X_tr = train.iloc[tr_idx][feature_cols]
    X_va = train.iloc[va_idx][feature_cols]
    y_tr = y_all[tr_idx]
    y_va = y_all[va_idx]

    m = lgb.LGBMRegressor(**lgb_params)
    m.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric='mae',
        callbacks=[lgb.early_stopping(300, verbose=False), lgb.log_evaluation(-1)],
    )
    oof_lgb[va_idx] = from_train_pred(m.predict(X_va))
    models_lgb.append(m)

    try:
        mx = xgb.XGBRegressor(**xgb_params)
        mx.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    except Exception:
        xgb_fb = dict(xgb_params)
        xgb_fb['objective'] = 'reg:squarederror'
        mx = xgb.XGBRegressor(**xgb_fb)
        mx.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    oof_xgb[va_idx] = from_train_pred(mx.predict(X_va))
    models_xgb.append(mx)

    mc = cb.CatBoostRegressor(**cat_params)
    mc.fit(
        X_tr, y_tr,
        eval_set=(X_va, y_va),
        verbose=max(1, cat_params['iterations'] // 5),
        use_best_model=True,
    )
    oof_cat[va_idx] = from_train_pred(mc.predict(X_va))
    models_cat.append(mc)

    fold_mae = mae(train.iloc[va_idx][TARGET].values, (oof_lgb[va_idx] + oof_xgb[va_idx] + oof_cat[va_idx]) / 3)
    print(f"\n  ✔ Fold {fold}  avg(M3) MAE {fold_mae:.4f}")

mae_lgb = mae(train[TARGET].values, oof_lgb)
mae_xgb = mae(train[TARGET].values, oof_xgb)
mae_cat = mae(train[TARGET].values, oof_cat)
print(f"\n▶ OOF MAE — LGB {mae_lgb:.6f} | XGB {mae_xgb:.6f} | CAT {mae_cat:.6f}")

model_maes = {'lgb': mae_lgb, 'xgb': mae_xgb, 'cat': mae_cat}
best_name = min(model_maes, key=model_maes.get)
worst_name = max(model_maes, key=model_maes.get)
best_mae = model_maes[best_name]
worst_mae = model_maes[worst_name]

use_models = ['lgb', 'xgb', 'cat']
if AUTO_DROP_WORST and worst_mae > best_mae * (1 + DROP_WORST_IF_GAP_GT):
    use_models.remove(worst_name)
    print(f"▶ drop worst model: {worst_name} (best={best_name})")

def weight(name):
    return 1.0 / (model_maes[name] ** WEIGHT_POWER)

w = {m: weight(m) for m in use_models}
w_sum = sum(w.values())

oof_ens = 0.0
if 'lgb' in use_models:
    oof_ens += w['lgb'] * oof_lgb
if 'xgb' in use_models:
    oof_ens += w['xgb'] * oof_xgb
if 'cat' in use_models:
    oof_ens += w['cat'] * oof_cat
oof_ens = oof_ens / w_sum

ens_mae = mae(train[TARGET].values, oof_ens)
print(f"▶ Ensemble OOF MAE (models={use_models}, p={WEIGHT_POWER}): {ens_mae:.6f}")


# ============================================================
# 5) Test 예측 + 제출
# ============================================================
section('Predict test + submit')
X_test = test[feature_cols]

p_lgb = np.mean([from_train_pred(m.predict(X_test)) for m in models_lgb], axis=0)
p_xgb = np.mean([from_train_pred(m.predict(X_test)) for m in models_xgb], axis=0)
p_cat = np.mean([from_train_pred(m.predict(X_test)) for m in models_cat], axis=0)

pred = 0.0
if 'lgb' in use_models:
    pred += w['lgb'] * p_lgb
if 'xgb' in use_models:
    pred += w['xgb'] * p_xgb
if 'cat' in use_models:
    pred += w['cat'] * p_cat
pred = pred / w_sum

pred = np.maximum(pred, CLIP_PRED_MIN)
pred_hi = float(np.percentile(train[TARGET].values, 100 * CLIP_PRED_MAX_Q))
pred = np.minimum(pred, pred_hi)

sub = pd.DataFrame({'ID': test['ID'], TARGET: pred})
save_path = os.path.join(project_root, 'submission_v10_base_only.csv')
sub.to_csv(save_path, index=False)
print(f"▶ saved -> {save_path}")
